In [ ]:
# Databricks notebook source
# tutorial_name: 02-Day9-Databricks-SQL-Warehouses-Dashboards
# notebookName: 02-Day9-Databricks-SQL-Warehouses-Dashboards
# COMMAND ----------

# DBTITLE 1,Paths
notepoint_rel = "hands-on/day-09/notebooks/02-Day9-Databricks-SQL-Warehouses-Dashboards.ipynb"
BASE_PATH = "abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/"
STUDENT_ID = "u25"
DAY5 = BASE_PATH + "/day5"
P_BASIC = DAY5 + "/delta/flight_summary_basic"
DAY9_ROOT = f"{BASE_PATH}day09-{STUDENT_ID}"
# COMMAND ----------

# DBTITLE 1,Prerequisite
try:
    spark.read.format("delta").load(P_BASIC).limit(1).collect()
    print("OK: P_BASIC")
except Exception as e:
    print("P_BASIC:", e)
# COMMAND ----------


# Day 9 — Item 24: Databricks SQL

Syllabus item 24 — **SQL warehouses** (Serverless vs **Pro**), **cost** (**auto-stop**), **dashboards** & **alerts**; hands-on **create SQL dashboard**.

Databricks SQL uses is a **separate web experience** from notebook clusters. This lab includes figures, **manual UI steps**, and **cluster-side** `%sql` / `spark.sql` on **`delta.`** paths so you can **prototype queries** here, then **paste** into a **SQL editor** for dashboards.

**Extra bundle:** `databricks/**/06-Spark-SQL.ipynb`, `**/M4-Spark-SQL.ipynb` for SQL patterns.


## Figure — SQL warehouse vs interactive cluster

```mermaid
flowchart LR
  subgraph nb[Notebook cluster]
    SP[Spark]
    SP --> ABFS[(abfss Delta)]
  end
  subgraph dbsql[Databricks SQL]
    WH[SQL warehouse]
    WH --> LC[Lakehouse tables / UC]
    WH --> Q[Queries Dashboards Alerts]
  end
```

**When to use SQL warehouse:** BI users, **low-latency** SQL, **serverless** elasticity, **governance** via UC.

**When to use notebook cluster:** PySpark, ML, **complex** ETL, **Jobs** you already built.


## Serverless vs Pro warehouses (concept)

| | Serverless (where available) | Pro |
|---|------------------------------|-----|
| Ops | Databricks scales | You size clusters |
| Cold start | Often faster | Configurable |
| Cost model | DBU + serverless rules | DBU classic |

**Manual:** **SQL** → **SQL warehouses** → **Create** → compare **Serverless** vs **Classic/Pro** per your region.


## Cost optimization — auto-stop

```text
  Warehouse idle minutes  --->  auto-stop  --->  $ saved
```

**Manual:** warehouse settings → **Auto stop** after **N minutes** (e.g. 10–30 for labs, higher for steady BI).

**Diagram**

```mermaid
flowchart LR
  Q[Queries running] -->|idle timer| STOP[Auto-stop]
  STOP -->|$| LOW[Lower spend]
```


## Hands-on — create a SQL dashboard (item 24) — **manual UI**

1. Sidebar **SQL** → **SQL Editor**.
2. Choose a **running warehouse** (start it if stopped).
3. New query — paste SQL that reads a **Unity Catalog** table **or** use a **managed** path your admin exposes.
4. **Run** → **Save** → **Add to dashboard** → new dashboard **layout** (tiles, refresh).
5. **Share** dashboard with group (viewer) per UC rules.

**If you only have ABFS paths:** register an **external table** or **managed table** in UC first (Day 4/5 patterns), then reference `catalog.schema.table` from SQL.


In [ ]:
# # Exercise (cluster): same query text you will paste into Databricks SQL (Delta path)
# Adjust path if your storage layout differs
q = f"SELECT DEST_COUNTRY_NAME, SUM(count) AS c FROM delta.`{P_BASIC}` GROUP BY 1 ORDER BY 2 DESC LIMIT 15"
print(q)
spark.sql(q).show(truncate=False)


## Alerts in Databricks SQL (manual)

1. Save a **query** that returns a single KPI (e.g. count > threshold).
2. **Create alert** → schedule **evaluation** → destination **email** / **webhook**.
3. On breach → notification fires.

Tie-in to **item 23**: same **alerting mindset** as **job failure** notifications.


## Figure — dashboard refresh loop

```mermaid
flowchart TB
  WH[Warehouse running]
  WH --> Q[Scheduled query]
  Q --> D[Dashboard tile]
  D --> U[Analysts]
  Q --> A[Alert if threshold]
```


In [ ]:
# Exercise: parameter-style filter (preview of what SQL widgets can do in UI)
_dest = "United States"
spark.sql(
    f"SELECT ORIGIN_COUNTRY_NAME, SUM(count) AS c FROM delta.`{P_BASIC}` WHERE DEST_COUNTRY_NAME = '{_dest}' GROUP BY 1 ORDER BY 2 DESC LIMIT 10"
).show(truncate=False)


## Checklist before you leave SQL workspace

- [ ] Warehouse **stopped** or **auto-stop** set
- [ ] Queries **saved** with clear names
- [ ] Dashboard **permissions** reviewed (no public over-share)
- [ ] Alerts **owned** by a group mailbox, not one person


## Next steps

- **03-Day9-Extras-Course-Review-and-Extensions.ipynb** — long supplement that recaps the whole course; use when time allows.


In [ ]:
# Optional cleanup
# dbutils.fs.rm(DAY9_ROOT, recurse=True)


## Extended manual — dashboard from scratch (12 steps)

1. **SQL** → **SQL Editor** → **+ New query**.  
2. **Attach** a warehouse → **Start** if needed.  
3. Write **SELECT** (start from cluster-tested SQL above).  
4. **Run** → fix errors until rows return.  
5. **Save** query with a name `vinsys_day9_top_dest`.  
6. **Add visualization** (bar / table).  
7. **Create dashboard** → name it `Day9_Capstone`.  
8. **Add** saved query tiles.  
9. **Arrange** layout (2×2 or full width).  
10. **Set refresh** (manual or schedule).  
11. **Share** with **Can View** for class group.  
12. **Screenshot** for your portfolio (optional).


## Figure — query lifecycle in SQL workspace

```mermaid
sequenceDiagram
  participant U as Analyst
  participant WH as Warehouse
  participant LC as Lakehouse table
  U->>WH: start warehouse
  U->>LC: SQL SELECT
  LC-->>U: result set
  U->>U: save query + dashboard
```


In [ ]:
# Exercise: second query template — top origins to one destination
spark.sql(
    f"SELECT ORIGIN_COUNTRY_NAME, SUM(count) AS c FROM delta.`{P_BASIC}` WHERE DEST_COUNTRY_NAME = 'United States' GROUP BY 1 ORDER BY 2 DESC LIMIT 12"
).show(truncate=False)


## Pitfall — SQL warehouse cannot see arbitrary `abfss` paths

Unless an admin registered **external locations** / **tables** in **Unity Catalog**, Databricks SQL expects **`catalog.schema.table`**. Use this notebook’s **`delta.\`path\``** pattern only on the **interactive cluster**; in **SQL warehouse**, migrate to **UC objects** first.


## ASCII — when to click which product

```text
  Engineer ETL / PySpark   ->  Notebook cluster + Jobs
  Analyst dashboard        ->  Databricks SQL + Warehouse
  Governed table ACLs      ->  Unity Catalog (Day 4)
```
